In [1]:
from dataclasses import dataclass
from enum import Enum
from typing import Any, Callable, Dict, Literal, Optional, Type
from pydantic import BaseModel, ConfigDict, Field, ValidationError


# ==========================================
# 1. 底层的 ToolRuntime 机制
# ==========================================
class ErrorKind(Enum):
    INVALID_ARGUMENT = "invalid_argument"  # Schema/Contract 校验失败
    RETRYABLE = "retryable"
    NON_RETRYABLE = "non_retryable"


@dataclass
class ToolError:
    kind: ErrorKind
    message: str
    retryable: bool


@dataclass
class ToolResult:
    ok: bool
    tool_name: str
    data: Any = None
    error: Optional[ToolError] = None


class ToolConfig:
    def __init__(self, name: str, schema_cls: Type[BaseModel]):
        self.name = name
        self.schema_cls = schema_cls


class ToolRegistry:
    def __init__(self):
        self.configs: Dict[str, ToolConfig] = {}
        self.funcs: Dict[str, Callable] = {}

    def register(self, config: ToolConfig, func: Callable):
        self.configs[config.name] = config
        self.funcs[config.name] = func

    def get_config(self, name: str) -> ToolConfig:
        return self.configs[name]

    def get_func(self, name: str) -> Callable:
        return self.funcs[name]


class ToolRuntime:
    """负责具体的防浪涌、隔离与可靠物理执行引擎。"""
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def execute(self, tool_name: str, validated_args: BaseModel) -> ToolResult:
        tool_fn = self.registry.get_func(tool_name)
        try:
            # 修正：此时参数已被 Schema 保证为【结构与类型契约合法】，
            # 但深层的业务规则与权限判断仍由后续 Policy/业务逻辑控制。
            raw_result = tool_fn(validated_args)
            return ToolResult(ok=True, tool_name=tool_name, data=raw_result)
        except Exception as e:
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                error=ToolError(kind=ErrorKind.NON_RETRYABLE, message=str(e), retryable=False),
            )


# ==========================================
# 2. Strict Tool Contract (配置 extra="forbid")
# ==========================================

# 统一配置严格模式：禁止任何未在 Schema 中显式定义的额外参数（Strict Extra Fields Guard）
STRICT_CONFIG = ConfigDict(extra="forbid")


class FindEmployeeByNameArgs(BaseModel):
    """用于通过姓名精确或模糊查询员工的基本 HR 信息。"""
    model_config = STRICT_CONFIG

    name: str = Field(
        ...,
        min_length=1,
        max_length=50,
        description="员工姓名，例如：'张三' 或 'John Doe'",
    )


class GetLeaveBalanceArgs(BaseModel):
    """用于查询特定员工当前的剩余带薪假期天数。"""
    model_config = STRICT_CONFIG

    employee_id: str = Field(
        ...,
        pattern=r"^EMP_\d{4}$",
        description="员工唯一编号，格式必须为 'EMP_' 加上 4 位数字，例如：'EMP_1001'",
    )


ExpenseCategory = Literal["travel", "meal", "hotel", "office"]


class CreateExpenseArgs(BaseModel):
    """
    提交一条报销申请。
    注意：本工具仅负责【创建/提交】申请，初始状态一律为 PENDING，不包含任何审批授权功能。
    """
    model_config = STRICT_CONFIG  # 显式禁止任何额外字段传入！

    employee_id: str = Field(
        ...,
        pattern=r"^EMP_\d{4}$",
        description="提交报销的员工编号，例如：'EMP_1001'",
    )
    amount: float = Field(
        ...,
        gt=0.0,
        le=50000.0,
        description="报销金额（人民币元），必须大于 0 且单笔不超过 50,000 元",
    )
    category: ExpenseCategory = Field(
        ...,
        description="报销类别，必须严格为以下枚举之一：'travel', 'meal', 'hotel', 'office'",
    )
    description: str = Field(
        ...,
        min_length=5,
        max_length=200,
        description="报销事由说明，不得少于 5 个字",
    )


# ==========================================
# 3. Agent Tool Dispatcher
# ==========================================
class AgentToolDispatcher:
    def __init__(self, registry: ToolRegistry, runtime: ToolRuntime):
        self.registry = registry
        self.runtime = runtime

    def dispatch(self, tool_name: str, raw_args: Dict[str, Any]) -> ToolResult:
        try:
            config = self.registry.get_config(tool_name)
        except KeyError:
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                error=ToolError(
                    kind=ErrorKind.INVALID_ARGUMENT,
                    message=f"Unknown tool name: '{tool_name}'",
                    retryable=False,
                ),
            )

        # 结构化契约校验
        try:
            validated_args = config.schema_cls.model_validate(raw_args)
        except ValidationError as e:
            error_details = "; ".join([f"{err['loc'][0]}: {err['msg']}" for err in e.errors()])
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                error=ToolError(
                    kind=ErrorKind.INVALID_ARGUMENT,
                    message=f"Contract Validation Failed -> {error_details}",
                    retryable=False,
                ),
            )

        return self.runtime.execute(tool_name, validated_args)


# ==========================================
# 4. Mock 业务函数定义与注册
# ==========================================
def mock_find_employee_by_name(args: FindEmployeeByNameArgs) -> dict:
    return {
        "status": "success",
        "employees": [
            {"employee_id": "EMP_1001", "name": args.name, "department": "IT Services"},
        ],
    }


def mock_get_leave_balance(args: GetLeaveBalanceArgs) -> dict:
    return {
        "status": "success",
        "employee_id": args.employee_id,
        "annual_leave_remaining_days": 7.5,
    }


def mock_create_expense(args: CreateExpenseArgs) -> dict:
    return {
        "status": "success",
        "expense_id": "EXP_20260911_0092",
        "employee_id": args.employee_id,
        "amount": args.amount,
        "category": args.category,
        "approval_status": "PENDING_APPROVAL",
    }


def setup_agent_system() -> AgentToolDispatcher:
    registry = ToolRegistry()

    registry.register(ToolConfig("find_employee_by_name", FindEmployeeByNameArgs), mock_find_employee_by_name)
    registry.register(ToolConfig("get_leave_balance", GetLeaveBalanceArgs), mock_get_leave_balance)
    registry.register(ToolConfig("create_expense", CreateExpenseArgs), mock_create_expense)

    runtime = ToolRuntime(registry)
    return AgentToolDispatcher(registry, runtime)


# ==========================================
# 5. 单元测试与边界验证
# ==========================================
if __name__ == "__main__":
    dispatcher = setup_agent_system()

    print("=========================================================")
    print("Test 1: 正常合法调用 (All Contracts Valid)")
    print("=========================================================")
    res1 = dispatcher.dispatch(
        "create_expense",
        {
            "employee_id": "EMP_1001",
            "amount": 350.0,
            "category": "meal",
            "description": "Team lunch meeting with client",
        },
    )
    print(f"Result OK: {res1.ok}")
    print(f"Data: {res1.data}")
    assert res1.ok
    assert res1.data["approval_status"] == "PENDING_APPROVAL"

    print("\n=========================================================")
    print("Test 2: 枚举越界拦截 (Invalid Category Enum)")
    print("=========================================================")
    res2 = dispatcher.dispatch(
        "create_expense",
        {
            "employee_id": "EMP_1001",
            "amount": 500.0,
            "category": "entertainment",
            "description": "KTV celebration party",
        },
    )
    print(f"Result OK: {res2.ok}")
    print(f"Error Message: {res2.error.message}")
    assert not res2.ok
    assert res2.error.kind == ErrorKind.INVALID_ARGUMENT

    print("\n=========================================================")
    print("Test 3: 正则格式与数值限制拦截 (Invalid ID RegEx & Over Amount)")
    print("=========================================================")
    res3 = dispatcher.dispatch(
        "create_expense",
        {
            "employee_id": "INVALID_ID_999",
            "amount": 999999.0,
            "category": "travel",
            "description": "Short trip",
        },
    )
    print(f"Result OK: {res3.ok}")
    print(f"Error Message: {res3.error.message}")
    assert not res3.ok
    assert res3.error.kind == ErrorKind.INVALID_ARGUMENT

    print("\n=========================================================")
    print("Test 4: 试图注入越权未知参数 (Extra Field Check -> MUST FAIL)")
    print("=========================================================")
    # LLM 试图自行填写 "approved": True，触发 extra="forbid" 强行拦截！
    res4 = dispatcher.dispatch(
        "create_expense",
        {
            "employee_id": "EMP_1001",
            "amount": 200.0,
            "category": "office",
            "description": "Purchase mechanical keyboard",
            "approved": True,  # 额外注入了未定义的参数
        },
    )
    print(f"Result OK: {res4.ok}")
    print(f"Error Message: {res4.error.message}")
    # 验证 Test 4 成功拦截！
    assert not res4.ok
    assert res4.error.kind == ErrorKind.INVALID_ARGUMENT
    assert "Extra inputs are not permitted" in res4.error.message

    print("\n✅ All Strict Contract Tests Passed Successfully!")

Test 1: 正常合法调用 (All Contracts Valid)
Result OK: True
Data: {'status': 'success', 'expense_id': 'EXP_20260911_0092', 'employee_id': 'EMP_1001', 'amount': 350.0, 'category': 'meal', 'approval_status': 'PENDING_APPROVAL'}

Test 2: 枚举越界拦截 (Invalid Category Enum)
Result OK: False
Error Message: Contract Validation Failed -> category: Input should be 'travel', 'meal', 'hotel' or 'office'

Test 3: 正则格式与数值限制拦截 (Invalid ID RegEx & Over Amount)
Result OK: False
Error Message: Contract Validation Failed -> employee_id: String should match pattern '^EMP_\d{4}$'; amount: Input should be less than or equal to 50000

Test 4: 试图注入越权参数 (Extra Undefined Parameter)
Result OK: True
Data (Notice 'approved' is completely ignored/filtered by Schema): {'status': 'success', 'expense_id': 'EXP_20260911_0092', 'employee_id': 'EMP_1001', 'amount': 200.0, 'category': 'office', 'approval_status': 'PENDING_APPROVAL'}

✅ All Contract Tests Passed Successfully!
